# Sampling Hard Circuits with Verifiably High Fidelity

#### **Reference paper:** [Sampling hard circuits with verifiably high fidelity](https://arxiv.org/abs/2607.25941)

---

Demonstrating quantum advantage requires two things: a task that is easy for a quantum computer but hard for a classical computer, and a way to verify the quantum computer's outputs. Both are hard on their own, and even harder together, since every trick that makes a circuit easy to certify tends to also make it easy to simulate.

<img src="./images/cover.png" width="800" />

The task we consider here is **random circuit sampling**. A quantum computer is given a random recipe of gates and asked to measure its qubits. The output distribution is not uniform noise. Instead, quantum interference makes some bitstrings far more likely than others, a "speckle pattern" or "quantum chaos" that is believed to be extraordinarily hard for any classical computer to reproduce.

The catch is that not every quantum circuit is classically hard. **Clifford circuits**, built from $H$, $S$, and CNOT gates, can be simulated efficiently on a laptop via the *stabilizer formalism*: each Clifford gate maps a Pauli operator to another Pauli, so an $n$ qubit state can be tracked by just $2n$ Pauli generators instead of a $2^n$ dimensional vector. What breaks this and pushes a circuit into the classically hard regime is the non-Clifford **$T$ gate**. A handful of $T$ gates on top of a large Clifford skeleton is enough to drive the system towards a Haar-random state, which, combined with the entanglement generated by the Clifford gates, is very hard to simulate. At the same time, the Clifford skeleton leaves behind Pauli symmetries that we can exploit to detect errors.

This is the idea behind **Doped Clifford Sampling (DCS)**, which rests on four ingredients:

1. **Spacetime Pauli check codes** for Clifford circuits.
2. **Direct Fidelity Estimation (DFE)** for Clifford circuits.
3. **$T$ doping** that preserves the validity of the spacetime codes.
4. **Fidelity lower bound estimation** for $T$ doped circuits.

In this notebook you will construct a $T$-doped circuit, derive a lower bound on its fidelity and certify the bound on hardware.

---

### Your task

Given a target Clifford circuit, you will work through the following steps, graded out of 100 points:

| Step | | Points | Where it runs |
|---|---|---:|---|
| 1 | Find a set of target qubits and define ancillas for spacetime Pauli checks. | 20 | local |
| 2 | Transpile and create a noise model for a specific device. | 5 | local |
| 3 | Insert spacetime Pauli checks with `paulice`. | 10 | local (longest classical step) |
| 4 | Choose the most effective ancilla qubits and reduce the number of checks to 10. | 10 | local |
| 5 | Compute post-selection rate for increasing number of checks using the QPU. | 10 | **QPU ≤ 10 s** |
| 6 | Compute Clifford circuit fidelity for increasing number of checks using the `Executor`. | 20 | local + **QPU ~9 min** |
| 7 | Dope circuit with $T$ gates, maintaining the structure needed for the checks. | 5 | local |
| 8 | Run resulting circuit on the QPU to obtain post-selection rates. | 5 | **QPU ≤ 10 s** |
| 9 | Estimate the fidelity lower bound of the resulting circuit. | 5 | local |
| 10 | [ADVANCED] Certify the 70-qubit circuit from the Quantum Advantage Tracker. | 10 | **QPU (leftover time)** |

**Total QPU budget: about 10 minutes for Steps 1–9.** Steps 5, 6 and 8 are the only ones that submit jobs. Every hardware cell is split in two: one cell **submits** and prints a job id, a second cell **retrieves** by id. Paste the ids into the configuration cell below so you can restart the kernel without re-running anything on the QPU.

### How the notebook is organized

Each step ends in a graded exercise. Cells are marked:

- **▶️ Run** — provided, just execute it.
- **✏️ Exercise** — you write the code where marked `# Add`.
- Helpers live in `utils.py`, which you can read and modify.

Along the way you will encounter colored boxes:

<div style="border-left: 5px solid #1ebb36; background: #ecfbef; color: #1a1a1a;
            padding: 12px 18px; border-radius: 4px; margin: 12px 0;
            box-sizing: border-box; max-width: 100%;
            overflow-wrap: break-word; word-break: break-word;">
<b>✅ Graded exercise</b><br>
An exercise whose result is checked by the grader and counts toward your final score.
</div>
<div style="border-left: 5px solid #e53935; background: #fdecea; color: #1a1a1a;
            padding: 12px 18px; border-radius: 4px; margin: 12px 0;
            box-sizing: border-box; max-width: 100%;
            overflow-wrap: break-word; word-break: break-word;">
<b>⚠️ QPU execution</b><br>
Important alerts and information about QPU access and hardware runs.
</div>
<div style="border-left: 5px solid #616161; background: #f4f4f4; color: #1a1a1a;
            padding: 12px 18px; border-radius: 4px; margin: 12px 0;
            box-sizing: border-box; max-width: 100%;
            overflow-wrap: break-word; word-break: break-word;">
<b>📖 Relevant references</b><br>
Pointers to sections of the paper, related work, or external documentation worth reading.
</div>
<div style="border-left: 5px solid #f57c00; background: #fff4e5; color: #1a1a1a;
            padding: 12px 18px; border-radius: 4px; margin: 12px 0;
            box-sizing: border-box; max-width: 100%;
            overflow-wrap: break-word; word-break: break-word;">
<b>🧪 Hints</b><br>
Suggestions and tips to help you make progress when you get stuck.
</div>

### Set up

Before you start:

- Create an IBM Cloud account and save your credentials.
- Clone the repository locally and create a Python virtual environment.
- Install `requirements.txt` (the `qc-grader` client, `paulice`, `samplomatic`, `qiskit-mitigation`, `mapomatic`).

In [ ]:
# ▶️ Run. Imports and the grading client (pip install qc-grader).

from pathlib import Path
import pickle
import time
from dataclasses import replace

import matplotlib.pyplot as plt
import numpy as np
from qiskit import ClassicalRegister, QuantumCircuit
from qiskit.quantum_info import Clifford, Pauli, PauliList
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService
from tqdm import tqdm

import utils
from qc_grader.challenges.qdc_2026 import (
    check_progress,
    grade_lab_verifiability_ex1,
    grade_lab_verifiability_ex2,
    grade_lab_verifiability_ex3,
    grade_lab_verifiability_ex4,
    grade_lab_verifiability_ex5,
    grade_lab_verifiability_ex6,
    grade_lab_verifiability_ex7,
    grade_lab_verifiability_ex8,
    grade_lab_verifiability_ex9,
    grade_lab_verifiability_ex10,
    join_team,
)


In [ ]:
# ✏️ Join your team before grading, with the name exactly as registered.
# Grading requests are refused until you join.
team_name = # Add your team name, e.g. "my-team"
join_team(team_name)


### Configuration

Everything that must stay fixed for grading, plus the job ids you collect along the way.
A safety switch controls QPU access: `SUBMIT_JOBS` guards every cell that spends QPU time. 

In [ ]:
# ▶️ Run. Configuration.

# --- Fixed for grading: do not change ---
num_qubits = 50  # data qubits in the target circuit
circuit_depth = 60  # CZ layers
PERIODIC = False  # False = open chain (graded). True = closed loop, needs an
# even num_qubits, and is not considered for grading.
seed = 2026
num_stabilizers = 100  # stabilizers sampled for DFE
num_T_gates = 100  # gates for T-doping
num_keep = 10  # checks kept after the Step 4 selection
reduced_checks = [0, 3, 7, 10]  # check counts run on the QPU in Step 6

# --- Shot budgets ---
shots_aer = 1000  # local simulation, cheap
shots_qpu = 1000  # Steps 5 and 8
shots_dfe_qpu = 5000  # Step 6 (qpu only, otherwise aer)
num_randomizations = 50  # Step 6 twirling

# --- Device ---
DEVICE = # Add "ibm_boston" or "ibm_phoenix"

# Everything this notebook saves or reloads lives here. mkdir because git does
# not track empty directories, and open(..., "wb") will not create one.
Path("saved_results").mkdir(exist_ok=True)

# --- QPU access ---
SUBMIT_JOBS = False  # True only when you intend to spend QPU time
JOB_IDS = {
    "step5_postselection": None,  # Job id
    "step6_readout": None,  # NoiseLearnerV3 job id
    "step6_dfe": None,  # Executor job id
    "step8_doped": None,  # Job id
}

print(f"{num_qubits} qubits, depth {circuit_depth}, seed {seed} | device {DEVICE}")
print(f"SUBMIT_JOBS = {SUBMIT_JOBS}")

### Note on grading

The grading functions evaluate different aspects of your solution. While you can try to bypass some steps, this notebook is designed to build progressively on previous results. Performing well at each stage is important to successfully complete the later tasks and obtain good results from the QPU.

Each graded cell calls a `grade_lab_verifiability_ex*` function from the `qc-grader` package, which sends your answer to the IBM Quantum challenge grading server and prints the result. Join your team first, in the cell after the imports. Only your score is stored, and the best score you reach on each step is the one that counts, so you can resubmit freely. Run `check_progress()` at any time to see your scores.

The grading server has no access to IBM Quantum and does not rerun your simulations. The grading functions read what they need from your objects on your machine and send it as you computed it, so the device calibration (Steps 1 and 2), the simulated post-selection rates (Step 4), the hardware results (Steps 5, 6, 8 and 10) and the bound built on them (Step 9) are **self-reported**. The server recomputes what it can from the circuits and numbers you send (gate counts, the noise model, the stabilizers of your circuit, the Clifford backbone of Step 10, the bound formula) and checks the rest for internal consistency. The organizers may audit the job ids you record in the configuration cell after the event.

---
### **Step 1.** Find a set of target qubits and define ancillas for spacetime Pauli checks

We start by creating the **random Clifford circuit** that plays the role of the target circuit throughout the notebook. It is generated from a *fixed seed* so everyone works with the same instance: a layer of Hadamards, then alternating even and odd layers of CZ gates, with a random single-qubit Clifford on each qubit touched by a CZ.

#### Open or periodic boundary conditions?

The chain can be left **open** or closed into a **loop**, and the choice matters for how hard the circuit is to simulate classically. The paper is explicit about this: an earlier version of the experiment used an open chain, and subsequent work exploited that geometry to contract the circuit as a tensor network in roughly 159 H100-GPU-hours, independently reproducing the measured fidelity. The published version therefore switched to periodic boundary conditions, which removes that low-width contraction path while still admitting plenty of ancillas.

In this notebook we work with the **open chain**, because picking effective checks is harder on the loop and we want the search to finish on a laptop. If you are curious, set `PERIODIC = True` above with an even `num_qubits` to close the loop, but that configuration is not considered for grading. For reference, the paper uses 64 qubits on a loop.

In [ ]:
# ▶️ Run. The fixed instance.


def _random_single_qubit_clifford(qc, q):
    """Random Clifford rotation from {S then SX} or {SX} on qubit `q`."""
    if np.random.random() < 0.5:
        qc.s(q)
    qc.sx(q)


def random_clifford_circuit(
    num_qubits: int, depth: int, periodic: bool = False
) -> QuantumCircuit:
    """Random Clifford on a 1D chain of `num_qubits` qubits.

    An H layer, then `depth` alternating even and odd layers of CZ gates, each CZ
    followed by a random single-qubit Clifford on both of its qubits.

    With `periodic=True` the wrap-around edge (num_qubits - 1, 0) joins the odd
    layers, closing the chain into a cycle. That only tiles the brickwork cleanly
    for an even number of qubits, so it is rejected otherwise.
    """
    if periodic and num_qubits % 2:
        raise ValueError(
            f"periodic boundary conditions need an even num_qubits, got {num_qubits}; "
            "an odd cycle cannot be covered by two alternating CZ layers"
        )
    qc = QuantumCircuit(num_qubits)
    qc.h(range(num_qubits))
    for d in range(depth):
        edges = [(i, i + 1) for i in range(d % 2, num_qubits - 1, 2)]
        if periodic and d % 2 == 1:
            edges.append((num_qubits - 1, 0))
        for a, b in edges:
            qc.cz(a, b)
            _random_single_qubit_clifford(qc, a)
            _random_single_qubit_clifford(qc, b)
    return qc


np.random.seed(seed)  # reset every time, so this cell is reproducible on re-run
circuit = random_clifford_circuit(num_qubits, circuit_depth, periodic=PERIODIC)

depth_2q = circuit.depth(lambda x: x.operation.num_qubits == 2)
print(f"{'periodic loop' if PERIODIC else 'open chain'} on {num_qubits} qubits")
print(f"gate counts: {dict(circuit.count_ops())}")
print(f"two-qubit depth: {depth_2q}")
circuit.draw("mpl", fold=-1, scale=0.6)

#### Choose a layout

Next, choose where the circuit will run on the device. Spacetime Pauli checks need free qubits next to some of the target (data) qubits. These neighbouring qubits act as ancillas that carry the check syndromes.

Layout choice has a large effect on the result. Pay particular attention to:

1. **Readout error.** Noisy ancilla readout can produce false syndromes and cause good shots to be discarded. Noisy data-qubit readout reduces the fidelity of the measured result.
2. **Two-qubit gate error.** Noisy gates between data qubits reduce the quality of the prepared state. Noisy gates used for checks can introduce errors and reduce the benefit of post-selection.

**Keep `layout` limited to data qubits.** It is passed to the transpiler as `initial_layout`, so it must contain exactly `num_qubits` entries. When comparing candidate layouts, however, consider the data qubits, their available ancillas, and the edges connecting them.

**Suggested workflow:**

1. **Find candidate layouts for the Clifford circuit.** Transpile the circuit using Qiskit's SABRE transpiler; this [tutorial](https://quantum.cloud.ibm.com/docs/en/tutorials/transpilation-optimizations-with-sabre) offers tips on using it. Extract the physical qubit indices from the transpiled circuit's `layout.final_index_layout()`. Alternatively, explore different candidates with mapomatic's [`matching_layouts(circuit, backend)`](https://github.com/qiskit-community/mapomatic).
2. **Find possible checks.** Call `get_check_qubits(backend, layout)`. It returns two aligned lists: each target qubit is paired with the ancilla at the same position in the other list.
3. **Reject defective pairs.** Use `utils.device_defects` with the readout and two-qubit error thresholds defined below. Keep a target–ancilla pair only if both qubits and their connecting edge are acceptable. Check that at least 15 pairs remain for Step 4; if not, try another layout or transpiler seed.
4. **Compare the remaining layouts.** You can pass a custom cost function to `mapomatic.evaluate_layouts(circuit, layouts, backend, cost_function)` to account for both data qubits and check ancillas. For a simple estimate of the couplers used here, you build a scoring circuit with one CZ on each neighbouring pair of data qubits and one CZ between each target and its ancilla. Add measurements to include readout error, then score it with the default cost function using `mapomatic.evaluate_layouts(scoring_circuit, layout + ancilla_qubits, backend)`. This per-layer estimate is easier to compare than an estimate for the full circuit depth.
5. **Inspect your choice.** Use `plot_coupling_map` to check the connections and `plot_error_map` to inspect their errors. `plot_coupling_map` requires [Graphviz](https://graphviz.org/download/) to be installed on your system.

This step is graded on layout quality, and the best achievable score depends on the device. Try different approaches and use the best layout you find for the rest of the lab. A full score may be difficult to achieve, so feel free to move on when your layout is good enough.

<div style="border-left: 5px solid #616161; background: #f4f4f4; color: #1a1a1a;
            padding: 12px 18px; border-radius: 4px; margin: 12px 0;
            box-sizing: border-box; max-width: 100%;
            overflow-wrap: break-word; word-break: break-word;">
<b>📖 Relevant references</b><br>
Reported calibration can be hours old. The most reliable approach is to re-benchmark the qubits you intend to use immediately before running, as described in
<a style="color: #0043ce" href="https://quantum.cloud.ibm.com/docs/en/tutorials/refresh-backend-properties-with-real-time-benchmarking">Refresh backend properties with real-time benchmarking</a>.
That needs a <code>Session</code> running the benchmark and the experiment back to back, which costs QPU time we do not have in this challenge, but it is the right thing to do in a real experiment, and worth reading now.

</div>

In [ ]:
# ▶️ Run. 1. Exclude broken hardware before searching for a layout.
# Keeps only qubits and couplers whose error is below 5e-2, and drops anything with
# no calibration data. Watch the kept counts: pruning too hard leaves a graph with
# no room for a chain of `num_qubits` qubits, since the long paths of a coupling map
# share qubits and each qubit removed cuts every path through it.
import mapomatic
from qiskit_paulice.layout import get_check_qubits

service = QiskitRuntimeService()
backend = service.backend(DEVICE)

defects = utils.device_defects(backend, ro_threshold=5e-2, tq_threshold=5e-2)
clean_edges = defects["clean_edges"]
assert clean_edges, "no hardware survived the cut; raise the thresholds"

<div style="border-left: 5px solid #f57c00; background: #fff4e5; color: #1a1a1a;
            padding: 12px 18px; border-radius: 4px; margin: 12px 0;
            box-sizing: border-box; max-width: 100%;
            overflow-wrap: break-word; word-break: break-word;">
<b>🧪 Hints</b>
<ul>
<li>Save your layout with <code>np.save</code> once you are happy, and reload it as <code>list(np.load(...))</code>.</li>
</ul>

</div>

In [ ]:
######### EXERCISE #########

layout = # Add layout (list of indices)

target_qubits, ancilla_qubits = # Add (use Paulice)

assert len(layout) == num_qubits, (f"layout must have {num_qubits} qubits, got {len(layout)}")
assert len(set(layout)) == num_qubits, "layout has repeated qubits"

# Overwrites any layout saved earlier. If this search found a good one, copy the
# file aside before re-running the cell, or the next search will replace it.
# np.save(f"saved_results/layout_{DEVICE}.npy", np.asarray(layout))
# layout = list(np.load(f"saved_results/layout_{DEVICE}.npy"))

# SUGGESTED:
# Prune ancillas/target qubits
# Check the cost using mapomatic
# score = #Add
# print(f"Layout score {score}")

print(f"Layout qubits ({len(layout)}): {layout}")
print(f"Target qubits  ({len(target_qubits)}):  {target_qubits}")
print(f"Ancilla qubits ({len(ancilla_qubits)}): {ancilla_qubits}")
print(
    f"\n{len(ancilla_qubits)} of {num_qubits} data qubits have an ancilla -> "
    f"at most {len(ancilla_qubits)} length-1 checks"
)

In [ ]:
# ▶️ Run. Your layout on the device.
#   green  = target (data) qubits
#   orange = ancilla qubits assigned to a check
#   grey   = unused
# plot_coupling_map needs the Graphviz system package (the `dot` program), which pip does not install:
#   macOS: brew install graphviz · Debian/Ubuntu: sudo apt-get install graphviz
#   Windows: winget install graphviz · conda: conda install -c conda-forge graphviz
# Restart the kernel after installing. If you'd rather not install Graphviz, skip this cell:
# it isn't graded, and plot_error_map below works without it.
from qiskit.visualization import plot_coupling_map

layout_set, ancilla_set = set(layout), set(ancilla_qubits)
plot_coupling_map(
    num_qubits=backend.num_qubits,
    qubit_coordinates=getattr(backend.configuration(), "qubit_coordinates", None),
    coupling_map=backend.configuration().coupling_map,
    figsize=(12, 12),
    qubit_color=[
        "#4CAF50" if i in layout_set else "#FF9800" if i in ancilla_set else "#DDDDDD"
        for i in backend.coupling_map.graph.node_indices()
    ],
    qubit_size=220,
    line_width=2,
    font_size=90,
)

In [ ]:
# ▶️ Run. Compare with backend calibration data.
from qiskit.visualization import plot_error_map

plot_error_map(backend)

<div style="border-left: 5px solid #1ebb36; background: #ecfbef; color: #1a1a1a;
            padding: 12px 18px; border-radius: 4px; margin: 12px 0;
            box-sizing: border-box; max-width: 100%;
            overflow-wrap: break-word; word-break: break-word;">
<b>✅ Grader — Step 1 · 20/100</b><br>
</div>

In [ ]:
grade_lab_verifiability_ex1(backend, layout, target_qubits, ancilla_qubits);

---
### **Step 2.** Transpile the circuit and build a noise model

Two things are needed before any check can be inserted:

1. An **ISA circuit**: the target circuit rewritten so the device can actually execute it. Transpilation translates every gate into that native basis of the device, and at the same time places the circuit on your chosen physical qubits so each 2-qubit gate lands on a real coupling edge.
2. A **noise model** of the device, used to *score* candidate checks and keep the ones that catch the most noise.

**The layout must survive transpilation.** Pass your `layout` as `initial_layout`. You can re-run the plotting cell above and confirm nothing moved.

For the noise model we use `NoiseModel.from_backend(..., uniform_gate_noise=True)`: one averaged error rate per gate type, taken from reported calibration. This is a simplification. A real experiment would **learn** the noise on the actual layers (`NoiseLearnerV3`), which gives a per-layer Pauli–Lindblad model and noticeably better check selection, but it costs QPU time we do not have here. Worth trying afterwards if you want better checks.

<div style="border-left: 5px solid #616161; background: #f4f4f4; color: #1a1a1a;
            padding: 12px 18px; border-radius: 4px; margin: 12px 0;
            box-sizing: border-box; max-width: 100%;
            overflow-wrap: break-word; word-break: break-word;">
<b>📖 Relevant references</b><br>
<code>paulice</code> provides
<a style="color: #0043ce" href="https://quantum.cloud.ibm.com/docs/en/api/qiskit-addon-paulice/qiskit-paulice-noise-models"><code>paulice.noise_models</code></a>
to build a noise model from backend calibration data or from a Pauli–Lindblad model.

</div>

In [ ]:
######### EXERCISE #########

from qiskit_paulice.noise_models import NoiseModel


def transpile_target(circuit, backend, layout):
    """Transpile `circuit` so that virtual qubit k lands on physical qubit layout[k]."""

    # Add

    return isa_circuit


def build_noise_model(backend, layout):
    """Build a paulice NoiseModel from `backend` calibration data."""

    # Add

    return noise_model


isa_circuit = transpile_target(circuit, backend, layout)
noise_model = build_noise_model(backend, layout)

print(f"ISA 2Q depth: {isa_circuit.depth(lambda x: len(x.qubits) == 2)}")
print(f"ISA gate counts: {dict(isa_circuit.count_ops())}")
print(noise_model)

<div style="border-left: 5px solid #1ebb36; background: #ecfbef; color: #1a1a1a;
            padding: 12px 18px; border-radius: 4px; margin: 12px 0;
            box-sizing: border-box; max-width: 100%;
            overflow-wrap: break-word; word-break: break-word;">
<b>✅ Grader — Step 2 · 5/100</b><br>

</div>

In [ ]:
grade_lab_verifiability_ex2(isa_circuit, noise_model, backend, layout);

---
### **Step 3.** Insert spacetime Pauli checks with `paulice`

<img src="./images/paulice.png" width="800" />

`qiskit_paulice.add_pauli_checks` searches for a good set of valid checks and returns a **family of circuits with increasing numbers of checks**, from zero up to one check per target qubit.

Key arguments:

* **circuit** *(positional)* — the Clifford circuit. With `stabilizers="all"` given it must contain **no** measurements; we add them later.
* **target_qubits** *(positional)* — data qubits to protect. Checks are added in this order, so the variant with $i$ checks uses `ancilla_qubits[:i]`.
* **noise_model** *(positional)* — used to score how much of the noise channel each check catches.
* **ancilla_qubits** — same length as `target_qubits`; `ancilla_qubits[i]` implements the check on `target_qubits[i]`.
* **cost** — which metric the search minimises:
  * `"gamma"`: the gamma value associated with the inverse logical noise channel, that is, the noise channel consisting of errors within the measurement lightcone that are undetectable by the checks.
  * `"LER"`: the empirical logical error rate after post-selection. Performs Monte Carlo simulations to calculate the fraction of post-selected shots affected by uncovered logical noise.
* **method** — `"windowed"` (random spatial windows over target-qubit wires), `"genetic"`, or `"windowed_genetic"`.
* **stabilizers** — which operators an error has to disturb before it counts as logical. With `stabilizers="all"` the checks are chosen modulo the final rotation basis of the measurement, which is what makes them reusable in Step 6: we will measure the payload in many different bases there to do direct fidelity estimation, and checks picked this way remain valid in all of them. It also fixes the right notion of "logical": an error counts as logical exactly when it changes the prepared state, which is precisely what fidelity measures.

**On the choices:** `windowed_genetic` generally finds better checks than `windowed`, at the cost of runtime. Try a few combinations of `cost` and `method` if you have time, then pick one and stick with it for the rest of the notebook. Each returned `CheckedCircuit` carries the achieved score in `.cost`, with `.cost_metric` naming which metric it is, so plot it against the number of checks: that curve is your first diagnostic, before any hardware is involved.

<div style="border-left: 5px solid #e53935; background: #fdecea; color: #1a1a1a;
            padding: 12px 18px; border-radius: 4px; margin: 12px 0;
            box-sizing: border-box; max-width: 100%;
            overflow-wrap: break-word; word-break: break-word;">
<b>⚠️ Classical runtime</b><br>
This is the longest classical step of the notebook. How long it takes depends on your laptop and on the search method and options you choose. The cell pickles its result, so you only pay once. You can use Qiskit Serverless if you want to run it on the cloud and free your laptop. See <code>serverless.py</code> for an example. To use it:
<ol style="margin: 6px 0 0 0; padding-left: 22px;">
<li>Install <code>qiskit-ibm-catalog</code> (it is in <code>requirements.txt</code>). It uses the same saved IBM Quantum account as the QPU steps.</li>
<li>In a new cell, run <code>import serverless</code> and then <code>serverless.upload()</code>. This sends <code>serverless/source_files/</code> to the cloud. You only need to do it once.</li>
<li>In the exercise cell below, call <code>serverless.add_checks(isa_circuit, target_qubits, ancilla_qubits, noise_model, seed=seed, ...)</code> wherever you would call <code>add_pauli_checks</code>. It takes the same keyword arguments.</li>
<li>Run the cell. It waits for the cloud job, prints the job's logs, and returns the same list, so no other cell changes. Step 4 calls the same <code>add_checks</code>, so it runs in the cloud too.</li>
</ol>
If your account is saved under a name, pass <code>name="..."</code> to both calls.
</div>

<div style="border-left: 5px solid #f57c00; background: #fff4e5; color: #1a1a1a;
            padding: 12px 18px; border-radius: 4px; margin: 12px 0;
            box-sizing: border-box; max-width: 100%;
            overflow-wrap: break-word; word-break: break-word;">
<b>🧪 Hints</b>
<ul>
<li>Draw <code>checked_circuits[-1].circuit</code> with <code>idle_wires=False</code> to confirm each check lands on the data qubit and ancilla you expect.</li>
<li>Each returned object is a <code>CheckedCircuit</code>, not a bare <code>QuantumCircuit</code>: it carries <code>.circuit</code>, <code>.target_qubits</code>, <code>.check_qubits</code>, <code>.check_support</code>, <code>.cost</code>, <code>.cost_metric</code>, <code>.get_postselection_method()</code>, <code>.dope()</code> and <code>.estimate_fault_rates()</code>. Keep the wrapper around, since the post-selection decoder lives on it.</li>
<li><code>.cost</code> and <code>.cost_metric</code> are plain attributes, not methods, so read them as <code>cc.cost</code> rather than <code>cc.cost()</code>. A cost curve that flattens tells you extra checks are buying nothing.</li>
</ul>

</div>

<div style="border-left: 5px solid #616161; background: #f4f4f4; color: #1a1a1a;
            padding: 12px 18px; border-radius: 4px; margin: 12px 0;
            box-sizing: border-box; max-width: 100%;
            overflow-wrap: break-word; word-break: break-word;">
<b>📖 Relevant references</b><br>
<a style="color: #0043ce" href="https://arxiv.org/abs/2504.15725">Sections II–IV of the supplementary information</a> describe how good spacetime Pauli checks are found.

</div>

In [ ]:
######### EXERCISE #########

from qiskit_paulice import add_pauli_checks

RECOMPUTE = True  # set False once the pickle exists


def add_checks(isa_circuit, target_qubits, ancilla_qubits, noise_model, **options):
    """Insert spacetime Pauli checks, returning one CheckedCircuit per check count."""

    # Add

    return checked_circuits


if RECOMPUTE:
    t0 = time.time()
    checked_circuits = add_checks(
        isa_circuit, target_qubits, ancilla_qubits, noise_model
    )
    print(f"found {len(checked_circuits)} checked circuits in {time.time() - t0:.0f}s")
    with open(f"saved_results/checked_circuits_{DEVICE}.pkl", "wb") as f:
        pickle.dump(checked_circuits, f)
else:
    with open(f"saved_results/checked_circuits_{DEVICE}.pkl", "rb") as f:
        checked_circuits = pickle.load(f)

print(f"Maximum number of checks: {len(checked_circuits) - 1}")
checked_circuits[-1].circuit.draw("mpl", fold=-1, idle_wires=False)

In [ ]:
######### EXERCISE #########

# Plot the achieved cost against the number of checks. A flattening curve means
# the extra checks are not catching much more of the noise channel.

check_costs = # Add

metric = checked_circuits[-1].cost_metric

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(
    range(len(check_costs)), check_costs, "o-", lw=1.5, color=utils.PALETTE["series"][0]
)
ax.set_xlabel("Number of checks")
ax.set_ylabel(f"{metric} (lower is better)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

<div style="border-left: 5px solid #1ebb36; background: #ecfbef; color: #1a1a1a;
            padding: 12px 18px; border-radius: 4px; margin: 12px 0;
            box-sizing: border-box; max-width: 100%;
            overflow-wrap: break-word; word-break: break-word;">
<b>✅ Grader — Step 3 · 10/100</b><br>
</div>

In [ ]:
grade_lab_verifiability_ex3(checked_circuits, ancilla_qubits);

---
### **Step 4.** Choose the most effective ancilla qubits and reduce the number of checks to 10

The **post-selection rate** is the fraction of shots that pass every check parity. It is the price of verification: more checks catch more errors, but fewer shots survive.

In practice, only a small number of carefully chosen checks is needed. For example, the experiment reported in the paper protects 64 data qubits using just 12 ancilla qubits. Motivated by this observation, we retain only num_keep = 10 checks in order to keep the computational cost of this notebook low. We identify potentially redundant checks by their effect on the post-selection rate: checks that never discard shots provide little additional filtering power. Accordingly, we rank checks by their impact on the post-selection rate under the noise model, assuming uniform error rates for all readout operations and two-qubit gates, and keep the most effective ones.

These are the steps that you need to follow:

1. **Simulate** every checked circuit with Aer and measure the post-selection rate.
2. **Rank** the individual checks by their effect on that rate.
3. **Select** `num_keep = 10`. You can use this step to re-evaluate the selected checks and discard any associated with poorly calibrated ancilla qubits or data-qubit-to-ancilla couplings on the target device.
4. **Re-run the search** restricted to those 10 ancillas, and compare the new rate curve with the first one.

First we need the payload measurements. The checked circuits measure only the ancillas, so we add a second classical register (`meas`) measuring the qubits in `layout`. A `CheckedCircuit` is a frozen dataclass, so `dataclasses.replace` swaps in the new circuit while keeping the post-selection decoder. That is what `utils.add_meas_register` does.

In [ ]:
# ▶️ Run. Add payload measurements, and build the Aer noise model.
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel as AerNoiseModel
from qiskit_aer.noise import ReadoutError, depolarizing_error

checked_circuits_sampling = [
    utils.add_meas_register(cc, layout) for cc in checked_circuits
]

# Same information as the paulice noise model used for check picking: a depolarizing
# channel on every CZ plus symmetric readout error. Single-qubit gates are noiseless,
# which is what lets the Step 6 basis change be free.
aer_nm = AerNoiseModel()
aer_nm.add_all_qubit_quantum_error(
    depolarizing_error(noise_model.gate_noise, 2), ["cz"]
)
p = noise_model.readout_noise
aer_nm.add_all_qubit_readout_error(ReadoutError([[1 - p, p], [p, 1 - p]]))
noisy_sim = AerSimulator(method="stabilizer", noise_model=aer_nm)

print(f"CZ depolarizing: {noise_model.gate_noise:.5f} | readout: {p:.5f}")
print(f"noisy instructions: {sorted(aer_nm.noise_instructions)}")

<div style="border-left: 5px solid #f57c00; background: #fff4e5; color: #1a1a1a;
            padding: 12px 18px; border-radius: 4px; margin: 12px 0;
            box-sizing: border-box; max-width: 100%;
            overflow-wrap: break-word; word-break: break-word;">
<b>🧪 Hint</b><br>
When <code>stabilizers='all'</code>, kept bitstrings are the ones where <code>checks_c == 0…0</code> directly. Instead, when we're finding checks with respect to terminal measurements, the check can still spread into the payload, and so you need to take the parity across multiple measurements. The <code>cc.get_postselection_method()</code> mask method works in either case.

</div>

In [ ]:
######### EXERCISE #########

# Simulate each checked circuit and compute the post-selection rate.
# Divide by the shots actually returned, not by `shots_aer`, so the number stays
# right if a run returns a different count.

aer_counts = []
for i, cc in enumerate(tqdm(checked_circuits_sampling, desc="Aer, all checks")):

    # Add: run `cc.circuit` on `noisy_sim` with shots=shots_aer and a per-i seed,
    #      and append the counts dict.

    pass

postselection_rate_aer = # Add: compute the post-selection rate for each checked circuit, as a list of floats.

utils.plot_postselection({"all checks (Aer)": postselection_rate_aer})

#### Which 10 checks to keep

The post-selection curve you just measured contains the ranking: each check is responsible for the step in the curve that appears when it is switched on, so comparing consecutive entries tells you what each individual check does to the rate. Calculate the drop between consecutive points, rank the checks by the size of that drop, and select the ancilla qubits for the ten largest drops.

<div style="border-left: 5px solid #f57c00; background: #fff4e5; color: #1a1a1a;
            padding: 12px 18px; border-radius: 4px; margin: 12px 0;
            box-sizing: border-box; max-width: 100%;
            overflow-wrap: break-word; word-break: break-word;">
<b>🧪 Hints</b>
<ul>
<li>Check that <code>postselection_rate_aer[0]</code> really is the zero-check circuit, where the rate must be exactly 1.0 because there is no syndrome to post-select on.</li>
</ul>

</div>

In [ ]:
######### EXERCISE #########

# Rank the checks by their effect on the post-selection rate and keep
# `num_keep` of them. Step 1 already excluded the bad hardware.

keep_checks = # Add: `num_keep` check indices, sorted ascending

assert len(keep_checks) == num_keep, f"only {len(keep_checks)} checks survived"

selected_ancilla_qubits = [ancilla_qubits[j] for j in keep_checks]
selected_target_qubits = [target_qubits[j] for j in keep_checks]

for j in keep_checks:
    print(
        f"  check {j:>3}  ancilla q{ancilla_qubits[j]:<4} "
        f"data qubit q{target_qubits[j]:<4}"
    )

Re-run the check search on just those 10 ancillas. This is not the same as slicing the old family: `add_pauli_checks` optimises the *set* of checks, so a search restricted to 10 ancillas might find better/worse individual checks than the first 10 of a 20-check solution.

<div style="border-left: 5px solid #e53935; background: #fdecea; color: #1a1a1a;
            padding: 12px 18px; border-radius: 4px; margin: 12px 0;
            box-sizing: border-box; max-width: 100%;
            overflow-wrap: break-word; word-break: break-word;">
<b>⚠️ Classical runtime</b><br>
The same search as Step 3 over a much smaller ancilla set, so it is faster. If needed, you can use Qiskit Serverless.
</div>

In [ ]:
######### EXERCISE #########

RECOMPUTE_SELECTED = True

if RECOMPUTE_SELECTED:
    t0 = time.time()
    selected_checked_circuits = # Add: add_checks on the selected target and ancilla qubits
    print(
        f"found {len(selected_checked_circuits)} checked circuits in {time.time() - t0:.0f}s"
    )
    with open(f"saved_results/selected_checked_circuits_{DEVICE}.pkl", "wb") as f:
        pickle.dump(selected_checked_circuits, f)
else:
    with open(f"saved_results/selected_checked_circuits_{DEVICE}.pkl", "rb") as f:
        selected_checked_circuits = pickle.load(f)

selected_cc_sampling = [
    utils.add_meas_register(cc, layout) for cc in selected_checked_circuits
]
print(f"Maximum number of checks: {len(selected_checked_circuits) - 1}")

In [ ]:
######### EXERCISE #########

# Same measurement as before, on the reduced set, then compare the two curves.

selected_postselection_rate_aer = # Add: compute the post-selection rate for each selected checked circuit, as a list of floats.

utils.plot_postselection(
    {
        "all checks (Aer)": postselection_rate_aer,
        "selected 10 checks (Aer)": selected_postselection_rate_aer,
    }
)

<div style="border-left: 5px solid #1ebb36; background: #ecfbef; color: #1a1a1a;
            padding: 12px 18px; border-radius: 4px; margin: 12px 0;
            box-sizing: border-box; max-width: 100%;
            overflow-wrap: break-word; word-break: break-word;">
<b>✅ Grader — Step 4 · 10/100</b><br>
</div>

In [ ]:
grade_lab_verifiability_ex4(
    selected_ancilla_qubits,
    selected_postselection_rate_aer,
    ancilla_qubits,
    postselection_rate_aer,
);

---
### **Step 5.** Compute post-selection rate for increasing number of checks using the QPU

In this step, you will perform the same measurement on the actual QPU. Expect the real rate to come out somewhat below the Aer curve: the uniform noise model gives every qubit and edge the same error rate, so it underestimates a real device with a spread of them. Nevertheless, it should not be far below. **If the rate collapses towards zero after one or two checks, stop, check for bugs and go back to Step 1**. That is the signature of a broken qubit or edge inside the layout, or of an ancilla whose readout is much worse than its reported calibration. No amount of work in later steps recovers from it, because every fidelity estimate from here on is computed from the shots that survive.


<div style="border-left: 5px solid #f57c00; background: #fff4e5; color: #1a1a1a;
            padding: 12px 18px; border-radius: 4px; margin: 12px 0;
            box-sizing: border-box; max-width: 100%;
            overflow-wrap: break-word; word-break: break-word;">
<b>🧪 Hint</b><br>
Retrieval here is a bit tricky. Two classical registers come back, <code>'meas'</code> for the payload and <code>'checks_c'</code> for the syndromes, but <code>get_postselection_method()</code> expects joined bitstrings in the right order. For this, you can use <code>result.join_data(names=[...])</code>. If the rate decreases too fast, it might be that you are not setting the right order.
</div>

<div style="border-left: 5px solid #e53935; background: #fdecea; color: #1a1a1a;
            padding: 12px 18px; border-radius: 4px; margin: 12px 0;
            box-sizing: border-box; max-width: 100%;
            overflow-wrap: break-word; word-break: break-word;">
<b>⚠️ QPU execution</b><br>
The full circuit family takes less than <b>10 seconds</b> of QPU time at 1,000 shots on a Heron r3 or Nighthawk r2 device. Submit all circuits together in one Sampler job. Remember to set <code>SUBMIT_JOBS = True</code> in the configuration cell.
</div>

In [ ]:
######### EXERCISE #########

from qiskit_ibm_runtime.executor_sampler import Sampler

if SUBMIT_JOBS:
    # Add: submit all circuits in selected_cc_sampling in a sampler job with shots=shots_qpu

    print(f"Job id: {job.job_id()}")
    JOB_IDS["step5_postselection"] = job.job_id()

else:
    print("SUBMIT_JOBS is False; skipping submission.")

In [ ]:
######### EXERCISE #########

# Retrieve by job id, assemble the joint bitstrings, and compute the rate.

job = service.job(JOB_IDS["step5_postselection"])
assert job.status() == "DONE", f"job {job.job_id()} is {job.status()}"

selected_counts_qpu = []
results = job.result()
for i, res in enumerate(results): 
    counts = #Add: extract counts from `res` and append to `selected_counts_qpu`
    selected_counts_qpu.append(counts)

selected_postselection_rate_qpu = # Add: compute the QPU post-selection rate for each selected checked circuit, as a list of floats.

utils.plot_postselection(
    {
        "selected 10 checks (Aer)": selected_postselection_rate_aer,
        "selected 10 checks (QPU)": selected_postselection_rate_qpu,
    }
)

<div style="border-left: 5px solid #1ebb36; background: #ecfbef; color: #1a1a1a;
            padding: 12px 18px; border-radius: 4px; margin: 12px 0;
            box-sizing: border-box; max-width: 100%;
            overflow-wrap: break-word; word-break: break-word;">
<b>✅ Grader — Step 5 · 10/100</b><br>
</div>

In [ ]:
grade_lab_verifiability_ex5(selected_postselection_rate_qpu);

#### Optional: try the Orbit Qiskit Function

The checks deal with noise by *detecting* it: they flag the bad shots and we throw them away. [Orbit](https://quantum.cloud.ibm.com/docs/en/guides/quantum-elements-orbit), a Qiskit Function from Quantum Elements, works on the same noise from the other side and *suppresses* it while the circuit runs, so fewer shots should get flagged. A single call combines three things:

- **Optimized transpilation.** We skip this part; see below.
- **Dynamical decoupling (DD).** Pulse sequences on idle qubits echo away dephasing and crosstalk while those qubits wait.
- **Measurement-error mitigation (MEM) with M3.** It needs none of the extra calibration jobs that a separate mitigation workflow would.

Step 5 uses only a few seconds of QPU time, so this is a cheap experiment. Run the same circuits through Orbit and compare the post-selection rate with your Step 5 curve. Does the gain grow with the number of checks, as more ancillas sit idle?

You send Sampler PUBs and use `pub_options` to choose how Orbit treats each one:

- `"raw"`: the baseline, with no DD and no mitigation;
- `"orbit"`: Orbit's defaults;
- `"custom"`: your own settings.

The defaults re-transpile the circuit, which would move the checks off the ancillas `paulice` placed them on. Our circuits are already ISA, so use `"custom"` with `"transpilation_mode": "validate"`. Orbit then keeps the layout, the routing and the check structure, and only adds DD and MEM:

```python
from qiskit_ibm_catalog import QiskitFunctionsCatalog

orbit = QiskitFunctionsCatalog(channel="ibm_quantum_platform").load("quantum-elements/orbit")
orbit_job = orbit.run(
    primitive="sampler",
    pubs=[(cc.circuit, None, shots_qpu) for cc in selected_cc_sampling],
    backend_name=DEVICE,
    # One entry applies to every PUB. Prepend {"mode": "raw"} PUBs for a baseline.
    options={"pub_options": [{"mode": "custom", "transpilation_mode": "validate", "mem": True}]},
)
print(orbit_job.job_id)
orbit_result = orbit_job.result()  # one PubResult per circuit, read as in the retrieve cell
```

When MEM succeeds, `get_counts()` returns the mitigated histogram. The unmitigated counts are in `pub_result.metadata["quantum_elements_orbit"]["measurementErrorMitigation"]["rawCounts"]`. The `register` field next to them tells you which classical register was mitigated, so check it before you join `'checks_c'` and `'meas'`.

<div style="border-left: 5px solid #e53935; background: #fdecea; color: #1a1a1a;
            padding: 12px 18px; border-radius: 4px; margin: 12px 0;
            box-sizing: border-box; max-width: 100%;
            overflow-wrap: break-word; word-break: break-word;">
<b>⚠️ QPU execution</b><br>
Optional and not graded. It uses <code>qiskit-ibm-catalog</code> (in <code>requirements.txt</code>). Qiskit Functions are an experimental preview, normally available only on the Premium, Flex and On-Prem plans. Your QDC challenge access includes them.
</div>

<div style="border-left: 5px solid #616161; background: #f4f4f4; color: #1a1a1a;
            padding: 12px 18px; border-radius: 4px; margin: 12px 0;
            box-sizing: border-box; max-width: 100%;
            overflow-wrap: break-word; word-break: break-word;">
<b>📖 Relevant references</b><br>
<a href="https://quantum.cloud.ibm.com/docs/en/guides/quantum-elements-orbit#example">Orbit guide, example</a>: a 50-bit Bernstein–Vazirani circuit on <code>ibm_kingston</code> gives 0 correct shots out of 4096 raw and 370 with Orbit, using the <code>"raw"</code> and <code>"orbit"</code> modes side by side in one job. Orbit helps most when circuits have long idle periods, so it is not guaranteed to help here.<br>
<a href="https://quantum.cloud.ibm.com/docs/en/api/functions/quantum-elements-orbit">Orbit API reference</a>: every option, including <code>dd_strategy</code>, <code>dd_qubits</code> and the result metadata.
</div>

---
### **Step 6.** Compute Clifford circuit fidelity for increasing number of checks

Post-selection rate tells us how many shots we keep, not whether the kept ones are any good. For that we need the **fidelity** of the noisy state $\rho$ against the ideal output $|\psi\rangle$.

Because $|\psi\rangle$ is a stabilizer state, we can use **Direct Fidelity Estimation (DFE)**. The fidelity is the average expectation value over the whole stabilizer group $\mathcal{S}$:

$$F = \mathrm{Tr}(\rho\,|\psi\rangle\langle\psi|) = \frac{1}{2^n} \sum_{G \in \mathcal{S}} \langle G \rangle_\rho$$

The group has $2^n$ elements, so for a large number of qubits it can neither be enumerated nor stored. Instead we sample $M$ elements uniformly and average:

$$\hat F_M = \frac{1}{M} \sum_{i=1}^M \langle G_i \rangle_\rho$$

A uniformly random **non-empty subset of the $n$ generators** is a uniformly random non-identity group element, which is what `utils.sample_stabilizers` uses. It returns the stabilizers *and* the `masks` recording which generators produced each one. Keep the masks, they are what make the next cell cheap.

In [ ]:
# ▶️ Run. Sample the stabilizers. `num_stabilizers` is yours to tune: more
# stabilizers means a tighter estimate at no extra shot cost, since they all come
# from the same measurements (see the next cell).

stabilizers, stab_masks = utils.sample_stabilizers(circuit, num_stabilizers, seed)

print(
    f"Sampled {len(stabilizers)} stabilizers of the {num_qubits}-qubit circuit "
    f"(two-qubit depth {depth_2q}):"
)
print(f"  {stabilizers[0]}, {stabilizers[1]}, ...")

#### Classically first

The stabilizer generators commute and can therefore be measured simultaneously. By applying $C^\dagger$ before measurement, where $C$ is the clifford circuit, the generator eigenvalues are extracted directly from computational-basis readout, allowing the expectation values of all stabilizers to be reconstructed from the same set of shots. As a result, increasing the number of stabilizers incurs almost no additional simulation cost.

This approach is practical in simulation, where the basis change is effectively noiseless, but not on hardware, where implementing $C^\dagger$ introduces significant gate overhead and error. Consequently, hardware experiments measure each stabilizer in its own basis and rely on additional error-mitigation techniques.

`utils.inverse_payload` builds $C^\dagger$; `utils.dfe_from_counts` does the sign bookkeeping. Complete the functions `measure_all_generators`and `dfe_sweep` as indicated below. You can check the input parameters in `utils.py` for clarity.

<div style="border-left: 5px solid #e53935; background: #fdecea; color: #1a1a1a;
            padding: 12px 18px; border-radius: 4px; margin: 12px 0;
            box-sizing: border-box; max-width: 100%;
            overflow-wrap: break-word; word-break: break-word;">
<b>⚠️ Classical runtime</b><br>
Depending on your laptop, this step can take a while. We would recommend to use a powerful laptop to run the classical DFE calculation.
</div>

In [ ]:
######### EXERCISE #########

inv_payload = utils.inverse_payload(circuit, aer_nm.noise_instructions)
print(f"basis change: {dict(inv_payload.count_ops())}")


def measure_all_generators(checked_circuit, inv_payload, layout):
    """Append the basis change and a payload measurement, keeping the checks intact.

    Returns a CheckedCircuit, so the post-selection decoder survives.
    """

    circ = checked_circuit.circuit.copy()
    circ.barrier()  # after everything else
    circ.compose(inv_payload, qubits=layout, inplace=True)  # payload qubits only

    # Add classical register to layout qubits

    return replace(checked_circuit, circuit=circ)


def dfe_sweep(circuits, label, stab_masks, inv_payload):
    """Return (F_raw, F_ps, F_group), one entry per checked circuit.

    utils.dfe_from_counts gives the three numbers:
      F_raw   -- over all shots
      F_ps    -- over the shots that pass the checks
      F_group -- P(all generator bits read 0) on the kept shots. This equals F_ps
                 exactly, so it is a free cross-check on the sampled estimate.
    """
    raw, ps, group = [], [], []
    for i, cc in enumerate(tqdm(circuits, desc=label)):
        # Add: measure_all_generators -> run on noisy_sim -> dfe_from_counts (check utils)

    return raw, ps, group


full_fid, full_fid_ps, full_fid_group = dfe_sweep(
    checked_circuits, "all checks", stab_masks, inv_payload
)
sel_fid, sel_fid_ps, sel_fid_group = dfe_sweep(
    selected_checked_circuits, "selected checks", stab_masks, inv_payload
)

In [ ]:
# ▶️ Run. Fidelity against check count, for both families.
# F_group should track F_ps closely; a gap means the 100-stabilizer sample is too
# small, or the sign bookkeeping is off.

utils.plot_fidelity_vs_checks(
    {
        "all checks, post-selected": full_fid_ps,
        "selected 10, post-selected": sel_fid_ps,
        "all checks, no post-selection": full_fid,
    },
    rates={
        "all checks": postselection_rate_aer,
        "selected 10": selected_postselection_rate_aer,
    },
    title="Clifford DFE fidelity (Aer)",
)

print("cross-check, selected family:")
for i, (a, b) in enumerate(zip(sel_fid_ps, sel_fid_group)):
    print(f"  {i:>2} checks   F_ps={a:.4f}   F_group={b:.4f}   diff={a - b:+.4f}")

#### Then on hardware

On the device we will need to implement other error mitigation techniques using the `Executor` stack:

- **TREX** (twirled readout error extinction) randomly flips each measured bit and undoes the flip in post-processing, which turns biased readout error into a rescaling factor that can be divided out. `NoiseLearnerV3` learns that factor first.
- **Non-Markovian error checks** add a post-circuit probe that catches faults that are not Pauli errors and that the spacetime checks were never designed to see. So there are now two independent post-selection conditions: the non-Markovian checks and the spacetime Pauli checks. Keep them as separate masks (`utils.non_markovian_mask`, `utils.check_mask`). 

**Important note:** Here we run only the four check counts in `reduced_checks` to stay inside the QPU budget.

<div style="border-left: 5px solid #f57c00; background: #fff4e5; color: #1a1a1a;
            padding: 12px 18px; border-radius: 4px; margin: 12px 0;
            box-sizing: border-box; max-width: 100%;
            overflow-wrap: break-word; word-break: break-word;">
<b>🧪 What to look for </b><br>
The post-selected fidelity should rise with the number of checks, while the un-post-selected one stays flat or drifts down: the checks do not make the state better, they identify the shots in which it already is. That contrast is the whole point of the experiment. With only a few check counts the trend is suggestive rather than conclusive. With more QPU time you would run the full set of eleven and see the curve properly.
</div>

<div style="border-left: 5px solid #e53935; background: #fdecea; color: #1a1a1a;
            padding: 12px 18px; border-radius: 4px; margin: 12px 0;
            box-sizing: border-box; max-width: 100%;
            overflow-wrap: break-word; word-break: break-word;">
<b>⚠️ QPU execution</b><br>
About <b>9 minutes</b> of QPU time: 4 check counts × 100 stabilizers × 50 randomizations at 100 shots each, plus a short readout-learning job. This is the single largest item in the budget. Check that the local DFE and the QPU postselection rates above look sensible first. 
</div>


In [ ]:
# ▶️ Run. Boxing, readout learning, and the measurement bases.
from qiskit.transpiler import PassManager
from qiskit_ibm_runtime import Executor, NoiseLearnerV3
from qiskit_ibm_runtime.quantum_program import QuantumProgram
from qiskit_mitigation.postselection import PostSelector
from qiskit_mitigation.postselection.passes import (
    AddPostCircuitNonMarkovianErrorChecks,
    AddSpectatorPostCircuitNonMarkovianErrorChecks,
)
from samplomatic.builders import build
from samplomatic.transpiler import generate_boxing_pass_manager
from samplomatic.utils import find_unique_box_instructions

# Generating passmanagers
boxing_pm = generate_boxing_pass_manager(measure_annotations="all")
add_non_markovian_checks = PassManager(
    [
        AddPostCircuitNonMarkovianErrorChecks("xslow"),
        AddSpectatorPostCircuitNonMarkovianErrorChecks(backend.coupling_map, "xslow"),
    ]
)

# Generating bases to measure. bases[i, 0, q] = basis for physical qubit q when measuring stabilizer i (Z=1, X=2, Y=3)
bases = utils.stabilizer_bases(stabilizers, layout, backend.num_qubits)

# Select reduced checked circuits
reduced_checked_circuits = [selected_checked_circuits[j] for j in reduced_checks]
print(f"running check counts {reduced_checks}")

# Submit readout learning job for TREX
if SUBMIT_JOBS:
    pm_meas = generate_preset_pass_manager(
        optimization_level=0, backend=backend, initial_layout=layout
    )
    measurement = boxing_pm.run(
        pm_meas.run(QuantumCircuit(num_qubits).measure_all(inplace=False))
    )
    readout_job = NoiseLearnerV3(backend).run(
        find_unique_box_instructions(
            measurement, normalize_annotations=None, undress_boxes=True
        )
    )
    print(f"readout learning job: {readout_job.job_id()}")
    JOB_IDS["step6_readout"] = readout_job.job_id()

<div style="border-left: 5px solid #f57c00; background: #fff4e5; color: #1a1a1a;
            padding: 12px 18px; border-radius: 4px; margin: 12px 0;
            box-sizing: border-box; max-width: 100%;
            overflow-wrap: break-word; word-break: break-word;">
<b>🧪 Defining QuantumProgram</b>
<ul>
<li>One <code>QuantumProgram</code> only. Append a samplex item per checked circuit. For each one: copy the circuit, add the <code>meas</code> register measuring <code>layout</code>, box it using <code>boxing_pm</code>, <code>build</code> it into a <code>(template, samplex)</code> pair, run the non-Markovian pass on the template, and save  <code>PostSelector.from_circuit(template, backend.coupling_map)</code> for later.</li>
<li><code>samplex.inputs().get_specs()</code> gives the basis input spec; feed it <code>bases[..., measured_qubits]</code> where <code>measured_qubits = sorted(layout + list(cc.check_qubits))</code>.</li>
</ul>

</div>

In [ ]:
######### EXERCISE #########

# Build one program item per reduced checked circuit, and keep the matching PostSelector.

program = QuantumProgram(shots=shots_dfe_qpu // num_randomizations)
selectors = []

for cc in reduced_checked_circuits:
    # Add meas register
    # Box the quantum circuit and built template and samplex
    # Add non markovian checks
    # selectors.append(PostSelector.from_circuit(...))

    # Obtain basis input from samplex
    # Obtain list of measured qubits

    # program.append_samplex_items with template, samplex, samplex_arguments and shape.


if SUBMIT_JOBS:
    dfe_job = Executor(backend).run(program)
    print(f"DFE job: {dfe_job.job_id()}")
    JOB_IDS["step6_dfe"] = dfe_job.job_id()

In [ ]:
# ▶️ Run. Retrieve the two jobs.

outs = service.job(JOB_IDS["step6_dfe"]).result()
(readout_map,) = service.job(JOB_IDS["step6_readout"]).result().to_pauli_lindblad_maps()

print(f"{len(outs)} variants retrieved")
print(f"mean readout error: {1 - np.mean(readout_map.probabilities()):.4f}")

In [ ]:
# ▶️ Run. Define basis map.

from qiskit.quantum_info import SparsePauliOp

observables = [SparsePauliOp(G) for G in stabilizers]
basis_map = {
    G: [obs if j == i else None for j, obs in enumerate(observables)]
    for i, G in enumerate(stabilizers)
}

assert len(basis_map) == len(stabilizers), (
    "duplicate stabilizers collapsed basis_map keys"
)
assert outs[0]["meas"].shape[0] == len(stabilizers), (
    "basis_map order does not match the submitted circuit order"
)

In [ ]:
# ▶️ Run. Define error mitigation reescaling factors and masks

from qiskit_mitigation.trex import TREX

# TREX
rescale = TREX.trex_factors_each_term(readout_map, observables)

# Non-markovian postselection
nm_masks = [
    utils.non_markovian_mask(out, sel) for out, sel in zip(outs, selectors, strict=True)
]

# Spacetime Pauli checks
check_masks = [utils.check_mask(out) for out in outs]


<div style="border-left: 5px solid #f57c00; background: #fff4e5; color: #1a1a1a;
            padding: 12px 18px; border-radius: 4px; margin: 12px 0;
            box-sizing: border-box; max-width: 100%;
            overflow-wrap: break-word; word-break: break-word;">
<b>🧪 Postprocessing</b><br>
<ul>

<li>The result <code>outs[k]</code>, defined above, is indexed by register name, with shape <code>(stabilizers, randomizations, shots, bits)</code>.</li>
<li><code>basis_map</code> above pairs each stabilizer with the basis it is measured in: <code>{G_i: [obs_i at position i, None elsewhere]}</code>. </li>
</ul>
</div>

In [ ]:
######### EXERCISE #########

from qiskit_mitigation.utils import executor_expectation_values

MIN_SHOTS_PER_BASIS = 20  # each stabilizer has its own basis; the thinnest one binds

def per_basis_shots(mask):
    """Shots surviving `mask` in each measurement basis (axis 0 = stabilizer)."""
    return mask.reshape(mask.shape[0], -1).sum(axis=1)

# Three nested stages. The two conditions stay separate arrays so either can be
# switched off alone. At 0 checks there is no checks_c register, so check_mask is
# all True and the last two stages coincide: expected, not a bug.
STAGES = ["TREX only", "+ non-Markovian PS", "+ check PS"]

qpu_fid = np.full((len(STAGES), len(reduced_checks)), np.nan)  # fidelities
qpu_err = np.full((len(STAGES), len(reduced_checks)), np.nan)  # std
qpu_keep = np.full((len(STAGES), len(reduced_checks)), np.nan)  # postselection rate

qpu_keep_checks = np.full(len(reduced_checks), np.nan) # Fraction of shots the spacetime checks reject 
qpu_shots_kept = np.zeros(len(reduced_checks), dtype=int) # Kept shots after spacetime checks and nm postselection
qpu_shots_total = np.zeros(len(reduced_checks), dtype=int) # Total number of shots per check count
qpu_shots_per_basis = np.zeros(len(reduced_checks), dtype=int) # Shots surviving in the thinnest measurement basis

for i, n_checks in enumerate(reduced_checks):

    nm, ck = nm_masks[i], check_masks[i]
    masks = [np.ones_like(nm), nm, nm & ck]  # one per stage, same order

    combined = nm & ck
    qpu_keep_checks[i] = ck.mean()
    qpu_shots_kept[i] = int(combined.sum())
    qpu_shots_total[i] = int(combined.size)
    qpu_shots_per_basis[i] = int(per_basis_shots(combined).min())

    for s, (label, mask) in enumerate(zip(STAGES, masks)):

        qpu_keep[s, i] = mask.mean()

        # Too few surviving shots in any one basis makes its expectation value
        # meaningless, so leave this variant as NaN rather than reporting noise.
        thinnest = int(per_basis_shots(mask).min())
        if thinnest < MIN_SHOTS_PER_BASIS:
            print(f"    skipped {label} at {n_checks} checks: "
                  f"{thinnest} shots in the thinnest basis")
            continue

        # The first two arguments are positional-only; the rest are keyword-only.
        evs = executor_expectation_values(
            # The measured payload bits (outs). Shape (stabilizers, randomizations, shots, bits),
            # Basis map defined above,
            meas_basis_axis=,   # which axis enumerates the measurement bases in the measured payload bits?
            avg_axis=,          # which axis holds the twirls in the measured payload bits?
            measurement_flips=, # check `outs` for measurement_flips. Twirling flipped some bits before readout.
            postselect_mask=,   # this STAGE's keep-mask, True = shot accepted
            rescale_factors=,   # TREX readout-mitigation factors, per Pauli term
        )

        qpu_fid[s, i], qpu_err[s, i] = utils.fidelity_from_evs(evs)

    print(f"  {n_checks:>2} checks  " + "  ".join(
        f"{lab}: {qpu_fid[s, i]:+.4f}±{qpu_err[s, i]:.4f}"
        for s, lab in enumerate(STAGES)))


In [ ]:
# ▶️ Run. Hardware fidelity against check count.
# Post-selected should rise with the checks; un-post-selected should not.
# The lower panel shows what that costs: the marginal check keep rate divides out
# the non-Markovian rejection, so it starts at 1.0 and is comparable to Step 5.

utils.plot_fidelity_vs_checks(
    {label: (qpu_fid[s], qpu_err[s]) for s, label in enumerate(STAGES)},
    rates={
        "non-Markovian": qpu_keep[1],
        "checks (marginal)": qpu_keep[2] / qpu_keep[1],
    },
    x=reduced_checks,
    # reference=("Aer, post-selected", sel_fid_ps),
    title="Clifford DFE fidelity on hardware",
)

Each additional check reduces the number of shots retained after non-Markovian postselection and spacetime Pauli checks. We use 5,000 shots, compared with 250,000 per stabilizer in the paper, so fidelity estimates become less reliable as more checks are added. Fidelity may initially improve, but eventually too few shots remain for a reliable estimate.

The next cell shows the percentage of shots retained after each filtering step.

In [ ]:
# ▶️ Run. What each post-selection condition costs on its own.

utils.plot_postselection(
    {
        "non-Markovian only": qpu_keep[1],
        "checks only": qpu_keep_checks,
        "both": qpu_keep[2],
    },
    x=reduced_checks,
    title="Post-selection rate on hardware, by condition",
)

print(f"{'checks':>7}{'shots kept':>13}{'of total':>12}{'rate':>9}{'per basis':>11}")
for i, k in enumerate(reduced_checks):
    print(
        f"{k:>7}{qpu_shots_kept[i]:>13,}{qpu_shots_total[i]:>12,}"
        f"{100 * qpu_shots_kept[i] / qpu_shots_total[i]:>8.2f}%"
        f"{qpu_shots_per_basis[i]:>11,}"
    )

<div style="border-left: 5px solid #1ebb36; background: #ecfbef; color: #1a1a1a;
            padding: 12px 18px; border-radius: 4px; margin: 12px 0;
            box-sizing: border-box; max-width: 100%;
            overflow-wrap: break-word; word-break: break-word;">
<b>✅ Grader — Step 6 · 20/100</b><br>

The grader also checks how you wired the Quantum program up, not only the fidelities it produced. Pass <code>program</code>, <code>circuit</code>, <code>layout</code>, <code>stabilizers</code> and <code>reduced_checked_circuits</code> along with your results; the grading function extracts what it needs from them.
</div>

In [ ]:
grade_lab_verifiability_ex6(
    qpu_fid,
    qpu_err,
    qpu_keep,
    reduced_checks,
    qpu_shots_per_basis,
    program,  # the QuantumProgram you ran
    circuit,  # the target circuit, untranspiled
    layout,
    stabilizers,
    reduced_checked_circuits,
);

---
### **Step 7.** Dope circuit with $T$ gates, maintaining the structure needed for the checks

Everything so far has been Clifford, and therefore classically easy. To make the sampling task classically hard we inject $T$ gates while keeping every spacetime Pauli check valid.

A $T$ gate commutes with $Z$ but not with $X$ or $Y$: $TZT^\dagger = Z$, while $TXT^\dagger$ becomes a superposition of $X$ and $Y$. So a $T$ is only safe at a spacetime location where the *only undetected* error is $Z$. Otherwise the $T$ rotates a harmless, undetected error into a harmful one the checks cannot catch.

`paulice` does the bookkeeping through `CheckedCircuit.dope()`, which inserts $R_Z(\pi/4)$ rotations. A site is valid only when **all** of the following hold:

1. **The forward-propagated doped $Z$-Pauli commutes with the $Z$-basis support of every check.** Otherwise the rotation would spoil a syndrome.
2. **It acts on the input state.** The back-propagated $Z$-Pauli is non-diagonal on at least one qubit, *or* it anticommutes with an earlier doped gate. A rotation that is diagonal all the way back to $|0\rangle^n$ does nothing.
3. **It affects the measured state.** Its forward-propagated $Z$-Pauli is non-diagonal on some qubit, *or* it anticommutes with a later doped gate. A rotation that stays diagonal to the end only contributes a phase.
4. **It is not a duplicate.** No earlier doped gate has the same forward-propagated $Z$-Pauli with only commuting doped gates in between. Two such rotations would merge into one.

Calling `dope()` with no arguments reports the full pool of valid sites. The pool shrinks as checks are added, since each new check removes $Z$-only sites, so the doping and the check count trade against each other directly.

Here, we will cap the count at `num_T_gates = 100`.

<div style="border-left: 5px solid #616161; background: #f4f4f4; color: #1a1a1a;
            padding: 12px 18px; border-radius: 4px; margin: 12px 0;
            box-sizing: border-box; max-width: 100%;
            overflow-wrap: break-word; word-break: break-word;">
<b>📖 Relevant references</b><br>
The check-commuting T-gate construction is described in <a style="color: #0043ce" href="https://arxiv.org/abs/2607.25941">Section S1 of the Supplementary Information</a>.

</div>

<div style="border-left: 5px solid #f57c00; background: #fff4e5; color: #1a1a1a;
            padding: 12px 18px; border-radius: 4px; margin: 12px 0;
            box-sizing: border-box; max-width: 100%;
            overflow-wrap: break-word; word-break: break-word;">
<b>🧪 Hints</b>
<ul>
<li><code>dope()</code> returns a <code>(QuantumCircuit, doped_wires)</code> tuple — a bare circuit, <b>not</b> a <code>CheckedCircuit</code>. The post-selection decoder does not travel with it. Wrap it back up with <code>dataclasses.replace(cc, circuit=doped)</code> so later steps can still post-select.</li>
<li>Use a fixed <code>seed</code> for every variant rather than <code>seed=j</code>, so all variants share one doping pattern. Otherwise each variant is a different ideal circuit, which confounds the comparison across check counts.</li>
</ul>

</div>

In [ ]:
######### EXERCISE #########

doped_checked_circuits = []

for j, cc in enumerate(selected_checked_circuits):
    # Add: report the full pool, then dope with num_sites=num_T_gates and a FIXED
    #      seed, and append a CheckedCircuit (not a bare circuit).

    # print(
    #    f"  {j:>2} checks: {len(all_wires):>4} valid sites, using {len(used_wires)}  "
    #    f"2Q depth {doped.depth(lambda x: len(x.qubits) == 2):>4}"
    # )


with open(f"saved_results/doped_circuits_{DEVICE}.pkl", "wb") as f:
    pickle.dump(doped_checked_circuits, f)

doped_checked_circuits[-1].circuit.draw("mpl", fold=-1, idle_wires=False)

<div style="border-left: 5px solid #1ebb36; background: #ecfbef; color: #1a1a1a;
            padding: 12px 18px; border-radius: 4px; margin: 12px 0;
            box-sizing: border-box; max-width: 100%;
            overflow-wrap: break-word; word-break: break-word;">
<b>✅ Grader — Step 7 · 5/100</b><br>
</div>

In [ ]:
grade_lab_verifiability_ex7(doped_checked_circuits, selected_checked_circuits);

---
### **Step 8.** Run resulting circuit on the QPU to obtain post-selection rates

This mirrors Step 5 on the doped circuits and it is a genuine experimental test: The $R_Z(\pi/4)$ rotations are implemented as **virtual frame changes**. The compiler absorbs them into the phases of later pulses, so no extra physical operation runs on the device and no extra noise is added. If that is true, the doped and Clifford post-selection rates must approximately coincide (modulo differences on the hardware at the time of execution). Two curves lying on top of each other is empirical evidence both that the $T$ gates are effectively noiseless and that the doping did not break the check structure.

Step 9 uses that assumption to bound the doped fidelity. If doping introduces no additional errors and only changes how existing errors propagate, the fidelity can only decrease when an error accepted by the Clifford checks becomes harmful in the doped circuit. The lower bound subtracts the probability of those events from the Clifford fidelity.

<div style="border-left: 5px solid #e53935; background: #fdecea; color: #1a1a1a;
            padding: 12px 18px; border-radius: 4px; margin: 12px 0;
            box-sizing: border-box; max-width: 100%;
            overflow-wrap: break-word; word-break: break-word;">
<b>⚠️ QPU execution</b><br>
The full circuit family takes less than <b>10 seconds</b> of QPU time at 1,000 shots on a Heron r3 or Nighthawk r2 device, as in Step 5.
</div>

In [ ]:
######### EXERCISE #########

# No transpilation needed: the checked circuits were already ISA, and dope() only
# inserts rz(pi/4), which is native. Just add the payload measurements.
doped_sampling = [utils.add_meas_register(cc, layout) for cc in doped_checked_circuits]

for i, cc in enumerate(doped_sampling):
    c = cc.circuit
    print(
        f"  {i:>2} checks  2Q depth {c.depth(lambda x: len(x.qubits) == 2):>4}  "
        f"rz {c.count_ops().get('rz', 0):>3}"
    )

if SUBMIT_JOBS:
    # Add: use sampler to submit a list of pubs with every circuit in `doped_sampling` with shots=shots_qpu

    print(f"Job id: {job.job_id()}")
    JOB_IDS["step8_doped"] = job.job_id()
else:
    print("SUBMIT_JOBS is False; skipping submission.")

In [ ]:
######### EXERCISE #########

job = service.job(JOB_IDS["step8_doped"])
assert job.status() == "DONE", f"job {job.job_id()} is {job.status()}"

doped_counts_qpu = []
results = job.result()
for i, res in enumerate(results):
    
    doped_counts = # Add: extract counts from `res` and append to `doped_counts_qpu`
        
doped_postselection_rate_qpu = # Add: compute the QPU post-selection rate for each doped checked circuit, as a list of floats.

utils.plot_postselection(
    {
        "Clifford (QPU)": selected_postselection_rate_qpu,
        "T-doped (QPU)": doped_postselection_rate_qpu,
    }
)

gap = np.abs(
    np.array(selected_postselection_rate_qpu) - np.array(doped_postselection_rate_qpu)
)
print(f"max |Clifford - doped| rate difference: {gap.max():.4f}")

<div style="border-left: 5px solid #1ebb36; background: #ecfbef; color: #1a1a1a;
            padding: 12px 18px; border-radius: 4px; margin: 12px 0;
            box-sizing: border-box; max-width: 100%;
            overflow-wrap: break-word; word-break: break-word;">
<b>✅ Grader — Step 8 · 5/100</b><br>
</div>

In [ ]:
grade_lab_verifiability_ex8(
    doped_postselection_rate_qpu, selected_postselection_rate_qpu
);

---
### **Step 9.** Estimate the fidelity lower bound of the resulting circuit

The doped circuit is not Clifford, so DFE no longer applies. Instead we use the paper's central result: the doped state's post-selected fidelity $F_2$ is bounded below by the Clifford fidelity $F_1$ measured in Step 6, minus one correction term:

$$F_2 \;\ge\; F_1 \;-\; \Pr\bigl(E \in H \,\big|\, E \in A\bigr).$$

$A$ is the set of Pauli faults **accepted** by the checks (they commute with every spacetime check), and $H \subseteq A$ is the subset that are **harmless** in the Clifford circuit because they stabilize the ideal Clifford output $|\psi_1\rangle$.

The intuition is the one Step 8 just tested: doping introduces no new errors, it only changes how existing ones propagate. A fault that was harmless in the Clifford circuit can become harmful once $T$ gates are present, and that is the *only* mechanism by which doping degrades fidelity.

<img src="./images/lower_bound.png" width="700" />

$\Pr(H \mid A)$ depends only on the Clifford circuit and the Pauli noise distribution, not on where the $T$ gates ended up, so it is computable classically. `CheckedCircuit.estimate_fault_rates` estimates it using a Monte Carlo procedure based on these steps:

1. **Draw a fault.** Every generator of the Pauli–Lindblad noise model fires independently, with probability $(1 - e^{-2\lambda})/2$ for its rate $\lambda$. 
2. **Accepted?** Propagate $E$ *forward* to the end of the circuit and compare it with the back-cumulant of the set of checks. Anticommuting with any one of them flips that syndrome bit, so the shot would have been thrown away by post-selection and it plays no further part.
3. **Harmless?** Propagate $E$ *backward* to the circuit input instead. $E$ stabilizes $|\psi_1\rangle = C_1|0^n\rangle$ exactly when $C_1^\dagger E C_1$ has no $X$ component on any qubit, since a pure $Z$-string acts on $|0^n\rangle$ as nothing more than a global phase.

$\Pr(H \mid A)$ is then the fraction of *accepted* shots whose fault is harmless and not the identity. The accompanying `harmless_stderr` is the binomial error on that fraction; the bound only says something when it is small next to the rate itself.

Two consequences worth keeping in mind. The propagation is Clifford, so this runs on the **undoped** checked circuits, which is all it ever needs. And the bound is deliberately pessimistic: it assumes every harmless fault turns maximally damaging once the $T$ gates are in, and it discards the harmful faults that doping can partly undo. The real $F_2$ therefore sits somewhere *above* the curve you plot.

<div style="border-left: 5px solid #616161; background: #f4f4f4; color: #1a1a1a;
            padding: 12px 18px; border-radius: 4px; margin: 12px 0;
            box-sizing: border-box; max-width: 100%;
            overflow-wrap: break-word; word-break: break-word;">
<b>📖 Relevant references</b><br>
Section S2 of the Supplementary Material of <a style="color: #0043ce" href="https://arxiv.org/abs/2607.25941">the paper</a> to see how these rates are computed.

</div>

In [ ]:
######### EXERCISE #########

harmless_rates, harmless_errs = [], []

for i, cc in enumerate(selected_checked_circuits):
    
     #Add

    # print(
    #    f"  {i:>2} checks  P(H|A) = {harmless_rate:.4f} "
    #    f"± {harmless_err:.4f}"
    # )

F_lower_qpu = qpu_fid[2] - np.array(harmless_rates)[reduced_checks]

utils.plot_bound(
    reduced_checks,
    F_lower_qpu,
    hardware=(qpu_fid[2], qpu_err[2]),
)

<div style="border-left: 5px solid #1ebb36; background: #ecfbef; color: #1a1a1a;
            padding: 12px 18px; border-radius: 4px; margin: 12px 0;
            box-sizing: border-box; max-width: 100%;
            overflow-wrap: break-word; word-break: break-word;">
<b>✅ Grader — Step 9 · 5/100</b><br>
</div>

In [ ]:
grade_lab_verifiability_ex9(harmless_rates, F_lower_qpu, qpu_fid, reduced_checks);

---
### **Step 10.[ADVANCED]** Certify a quantum-advantage-scale circuit

Everything up to here ran on a circuit this notebook built for itself, at a size chosen so that every step finishes in a reasonable amount of time. The method, though, is the one behind real quantum advantage claims. This last step points you at one of those claims and asks you to reproduce its certificate.

The [Quantum Advantage Tracker](https://quantum-advantage-tracker.github.io/) collects candidate demonstrations together with the data needed to check them. Under **Classically verifiable problems**, find the circuit model `doped_random_graph_sampling`, instance **`nq70_depth70_checks27`** (contributed by IBM and the University of Chicago). It is this notebook's construction, scaled up:

|                            | this notebook | tracker instance |
| -------------------------- | ------------: | ---------------: |
| data qubits                |            50 |           **70** |
| CZ depth                   |            60 |           **70** |
| $T$ gates                  |           100 |          **468** |
| ancillas used for checks   |            10 |           **27** |
| best known Schmidt rank    |             — |         $2^{30}$ |
| best known stabilizer rank |             — |      $2^{185.5}$ |

The last two rows are what make it a candidate rather than an exercise. A Schmidt rank of $2^{30}$ is the bond dimension a tensor network would need, and a stabilizer rank of $2^{185.5}$ is the term count a Clifford + $T$ decomposition would need. Both are out of reach, so no known classical method samples this distribution.

#### The task

Download the circuit **without checks** from the instance folder:

- [`nq70_depth70_checks27_doped.qasm`](https://raw.githubusercontent.com/quantum-advantage-tracker/quantum-advantage-tracker.github.io/main/data/classically-verifiable-problems/circuit-models/doped_random_graph_sampling/nq70_depth70_checks27_doped.qasm) — 70 qubits, the doped graph-state preparation. **This is the one you want.**
- `nq70_depth70_checks27_doped_checks.qasm` — the same circuit with the contributors' own 27 checks already inserted on 27 ancillas, 97 qubits in total. Leave this one alone until you want to compare notes at the end.

Both live in [the instance folder on GitHub](https://github.com/quantum-advantage-tracker/quantum-advantage-tracker.github.io/tree/main/data/classically-verifiable-problems/circuit-models/doped_random_graph_sampling), alongside a `README.md` that documents the construction and the verifiability argument in full.

For reference, the tracker reports a state fidelity of **0.26 ± 0.02** for the contributors' 27-check run, conservatively bounded above 0.036 with 95% confidence. Your target is more modest: apply this notebook's workflow to the circuit and **certify a fidelity lower bound above 0.2**.

How many checks to insert is yours to decide, and that choice is the real content of the exercise: each extra check buys fidelity and costs post-selection rate. One data point is enough. Report the number of checks you settled on and the bound you reached with it.


#### How to go about it

The steps are the ones you have already done, with two wrinkles at the start.

1. **Load the QASM.** `qiskit.qasm2.load` will reject the file with `'sx' is not defined in this scope`, because `sx`, `sxdg` and `s` are not part of `qelib1.inc`. Pass Qiskit's legacy set instead:

   ```python
   from qiskit import qasm2

   circuit = qasm2.load(path, custom_instructions=qasm2.LEGACY_CUSTOM_INSTRUCTIONS)
   ```

   What comes back has no measurements and no classical register, so add your own payload register as in Step 5.

2. **Recover the Clifford backbone.** The bound rests on $F_2 \ge F_1 - \Pr(H \mid A)$, and $F_1$ is the fidelity of the *undoped* Clifford state, measured by DFE. Here the doping is exactly the 468 `rz(pi/4)` instructions and nothing else, so dropping them, or setting their angle to zero, leaves the graph-state preparation behind.

3. **Layout.** You need a 70-qubit path with enough dangling ancillas beside it. Step 1 applies unchanged, but choosing well matters even more here than it did above. Figure 3 of the instance `README.md` shows the layout the contributors used on a Heron device, if you want a reference. The most reliable approach is to re-benchmark the qubits you intend to use immediately before the DFE run, inside a single `Session`, as described [here](https://quantum.cloud.ibm.com/docs/en/tutorials/refresh-backend-properties-with-real-time-benchmarking), but this uses about $3$ min of additional QPU time.

4. **Checks, then the bound.** Insert your own checks on the Clifford circuit (Steps 3 and 4), choose how many to keep, measure $F_1$ by DFE (Step 6), compute $\Pr(H \mid A)$ with `estimate_fault_rates` (Step 9), and subtract.

5. **What to skip.** Steps 7 and 8. The circuit arrives already doped, so there is nothing to dope; re-running the doped family on hardware would cost QPU time this step does not have, and inserting checks into an already-doped circuit would restrict where those checks can go. So work with the Clifford backbone and report a bound from Steps 1–4, 6 and 9. 

**What is needed for the grader.** Six numbers and one circuit: `num_checks`, `F_1` and `F_1_err`, `harmless_rate`, `F_lower`, `num_stabilizers` and `shots_per_basis`, plus `clifford_backbone`, which is the circuit exactly as loaded from the QASM with its 468 `rz(pi/4)` removed, in virtual-qubit order, before any transpilation. That last one is how the grader confirms you certified *this* circuit; it also checks your uncertainty against what your shot count can support, so `shots_per_basis` reports the shots you actually kept.

<div style="border-left: 5px solid #f57c00; background: #fff4e5; color: #1a1a1a;
            padding: 12px 18px; border-radius: 4px; margin: 12px 0;
            box-sizing: border-box; max-width: 100%;
            overflow-wrap: break-word; word-break: break-word;">
<b>⚠️ Hints</b>
<ul>
<li><b>Budget the QPU time before you submit anything.</b> A mock backend gives you an estimate for free. For scale: the contributors keep 0.038% of their shots with 27 checks, and every check beyond that makes it smaller. What sets the error bar on the fidelity is the number of shots surviving <i>per measurement basis</i>, not the number you submitted, so watch both as you vary the check count and expect to need more shots than Step 6 used.</li>
</ul>
</div>


In [ ]:
######### EXERCISE #########

# Add your code here


<div style="border-left: 5px solid #1ebb36; background: #ecfbef; color: #1a1a1a;
            padding: 12px 18px; border-radius: 4px; margin: 12px 0;
            box-sizing: border-box; max-width: 100%;
            overflow-wrap: break-word; word-break: break-word;">
<b>✅ Grader — Step 10 · 10/100</b><br>
</div>

In [ ]:
grade_lab_verifiability_ex10(
    clifford_backbone,  # the QASM circuit with its 468 rz(pi/4) removed, untranspiled
    num_checks,  # how many checks you kept
    F_1,  # post-selected Clifford fidelity from the QPU
    F_1_err,  # and its uncertainty
    harmless_rate,  # Pr(H | A), from estimate_fault_rates
    F_lower,  # your bound, F_1 - Pr(H | A)
    num_stabilizers,  # stabilizers sampled for the DFE
    shots_per_basis,  # kept shots in the thinnest measurement basis
);

In [ ]:
# ▶️ Run. Your scores, as stored by the grading server.
check_progress()

---
### Closing note

If you didn’t reach Step 10, don’t worry. That step was intentionally ambitious. The key learning objective was the method itself, which you developed through Step 9, and that is what this notebook was designed to teach.

If you did make it to Step 10, congratulations! You applied the method to certify a real entry in the Quantum Advantage Tracker, a recent result at a scale where no known classical method can independently verify the answer.

Either way, you have learned how to do something genuinely challenging: evaluate the performance of a quantum circuit whose output cannot be classically verified, and still assign a meaningful and trustworthy measure of its quality.

Thank you very much for participating!


### You've reached the end of the notebook — congratulations! 🎉

#### References

1. [Sampling hard circuits with verifiably high fidelity](https://arxiv.org/abs/2607.25941), arXiv:2607.25941.
2. [Spacetime Pauli checks](https://arxiv.org/abs/2504.15725), arXiv:2504.15725.
3. [`qiskit-paulice` documentation](https://quantum.cloud.ibm.com/docs/en/api/qiskit-addon-paulice/qiskit-paulice-noise-models).
4. [Refresh backend properties with real-time benchmarking](https://quantum.cloud.ibm.com/docs/en/tutorials/refresh-backend-properties-with-real-time-benchmarking).